In [1]:
import os
import re
from ase.io import read
from ase import Atoms
import pandas as pd
import numpy as np
import random as r
import sys
from collections import defaultdict, deque

def near(atomnum, atoms, tolerance=2.000, Cfirst=True):
    atoms_not_H = []
    for i, atom in enumerate(atoms):
        if atom.symbol != 'H':
            atoms_not_H.append(i)
    mole = [atomnum]
    pre_distances = atoms.get_distances(atomnum, atoms_not_H)
    distances = np.delete(pre_distances, np.where(pre_distances >= tolerance))
    distances = np.delete(distances, np.where(distances == 0))
    distances.sort()
    length = len(distances)
    if length > 4:
        distances = distances[:4]
    for i, atom in enumerate(atoms_not_H):
        distance = atoms.get_distance(atom, atomnum)
        if distance in distances:
            mole.append(atom)
    mole_C = []
    mole_other = []
    for num in mole:
        if atoms[num].symbol == 'C':
            mole_C.append(num)
        else:
            mole_other.append(num)
    if len(distances) == 1:
        if atoms[atomnum].symbol == 'C':
            return mole_C + mole_other
        else:
            return mole_other + mole_C
    if Cfirst == True:
        return mole_C + mole_other
    else:
        return mole_other + mole_C

def allone(lists):
    judge_box = []
    for ls in lists:
        for i in ls:
            judge_box.append(i)
    if 0 in judge_box:
        return 0
    else:
        return 1
    
def num_in_list(ls, a):
    num = 0
    for i in ls:
        if i == a:
            num += 1
    return num

def allroutes(file, loc='./', atomnums=[0,1], tolerance=2.000,\
              form='XYZ', Cfirst=True, ignores=[], must=[], rangelimit=[], spiropoints=[], verbose=False):
    passeds = []
    opath = os.getcwd()
    os.chdir(loc)
    atoms = read(file+'.xyz')
    atomnum1 = atomnums[0]
    atomnum2 = atomnums[1]
    atoms_not_H = []
    atoms_H = []
    for i, atom in enumerate(atoms):
        if atom.symbol != 'H':
            atoms_not_H.append(i)
        else:
            atoms_H.append(i)
    if spiropoints != []:
        spiropoint_dict = {}
        for point in spiropoints:
            spiropoint_dict[point] = 0
    
    #Trace Back
    step = 1
    z = 1
    i = 1
    passed = []
    routes = []
    real_moles = []
    route = [atomnum1]
    tt = 0
    while 1:
        mole = near(atomnum1, atoms, tolerance=tolerance, Cfirst=Cfirst)
        if verbose == True:
            print('Near:', mole)
        real_mole = [*mole]
        real_mole.remove(atomnum1)
        real_moles.append(real_mole)
        passed = [0]*len(real_mole)
        passeds.append(passed)
        
        if verbose == True:
            print(1)
            print('Real Near:', real_mole)
            print('Passed:', passeds)
            print('Number:', real_moles)
            print('Route:', route)
            print()
        
        for i,p in enumerate(passeds[step-1]):
            if real_moles[step-1][i] in ignores:
                passeds[step-1][i] = 1
            if real_moles[step-1][i] in route:
                atomnum3 = real_moles[step-1][i]
                if spiropoints != []:
                    if atomnum3 in spiropoints:
                        #if spiropoint_dict[atomnum3] == 0:
                        #    spiropoint_dict[atomnum3] = 1
                        #elif spiropoint_dict[atomnum3] == 1:
                        #    passeds[step-1][i] = 1
                        pass
                    else:
                        passeds[step-1][i] = 1
                else:
                    passeds[step-1][i] = 1
        
        #if len(route) >= 2 and route[-2] == atomnum1:
        #    passeds[step-1][j] = 1
        #    continue
        
        if allone(passeds) == 1:
            break
        
        if spiropoints != []:
            for num in route:
                while num_in_list(route, num) > 2:
                    del passeds[-1]
                    step = step - 1
                    del route[-1]
                    del real_moles[-1]
            
            if len(route) >= 3:
                if route[-1] == route[-3]:
                    tt = 1
            if tt == 1:
                del passeds[-1]
                del passeds[-1]
                step = step - 2
                del route[-1]
                del route[-1]
                del real_moles[-1]
                del real_moles[-1]
                tt = 0
        
        #print(step)
        for passed0 in passeds:
            if 0 in passed0:
                tt = 1
        if tt == 0:
             break
        else:
            tt = 0
            while 0 not in passeds[step-1]:
                del passeds[-1]
                step = step - 1
                del route[-1]
                del real_moles[-1]
        
        for j, p in enumerate(passeds[step-1]):
            if p == 0:
                break
        
        atomnum1 = real_moles[step-1][j]
        #if spiropoints != []:
        #    if atomnum1 in spiropoints:
        #        if spiropoint_dict[atomnum1] == 0:
        #            spiropoint_dict[atomnum1] = 1
        #        elif spiropoint_dict[atomnum1] == 1:
        #            passeds[step-1][j] = 1
        #    else:
        #        passeds[step-1][j] = 1
        #else:
        passeds[step-1][j] = 1
        
        route.append(atomnum1)
        if verbose == True:
            print(2)
            print('Real Near:', real_moles[step-1])
            print('Passed:', passeds)
            print('Number:', real_moles)
            print('Route:', route)
            print()
        
        if route[-1] == atomnum2:
            mustnum = 1
            rangenum = 1
            if must != []:
                for num in must:
                    if num not in route:
                        mustnum = 0
            if rangelimit != []:
                for num in route:
                    if num not in rangelimit:
                        mustnum = 0
            if mustnum == 1 and rangenum == 1:
                print('\nCorrect Route:', route)
                ox = 1
                ox2 = 1
                if spiropoints != []:
                    for spiropoint in spiropoints:
                        if num_in_list(route, spiropoint) > 2:
                            ox = 0
                    
                    for k, _ in enumerate(route):
                        if route[k] == route[k-2]:
                            ox2 = 0
                    if ox == 1 and ox2 == 1:
                        routes.append([*route])           
                else:
                    routes.append([*route])
                print('\n')
        
        step = step + 1

        z = z + 1
        
        if z > 99999:
            break
    if verbose == True:
        print(z, 'steps to find these routes.')
        print('Routes:', routes)
    os.chdir(opath)
    return routes

def routelength(file, a, loc='./'):
    opath = os.getcwd()
    os.chdir(loc)
    atoms = read(file+'.xyz')
    distance = 0
    for i in range(len(a)-1):
        x1 = a[0]
        x2 = a[1]
        distance = distance + atoms.get_distances(x1, x2)[0]
    os.chdir(opath)
    return distance

In [2]:
def near_notH(atomnum, atoms, tolerance=1.7):
    atoms_not_7 = []
    for i, atom in enumerate(atoms):
        if atom.symbol not in ['H', 'F', 'Cl', 'Br']:
            atoms_not_7.append(i)
    mole = []
    mole_ele = []
    pre_distances = atoms.get_distances(atomnum, atoms_not_7)
    distances = np.delete(pre_distances, np.where(pre_distances == 0))
    distances.sort()
    length = len(distances)
    distances = distances[:4]
    for i, atom in enumerate(atoms_not_7):
        distance = atoms.get_distance(atom, atomnum)
        if distance in distances:
            ele1 = atoms.get_chemical_symbols()[atom]
            ele2 = atoms.get_chemical_symbols()[atomnum]
            if 'S' in [ele1, ele2]:
                if distance < tolerance + 0.45:
                    mole.append([atomnum, atom])
                    mole_ele.append([ele2, ele1])
            else:
                if distance < tolerance:
                    mole.append([atomnum, atom])
                    mole_ele.append([ele2, ele1])
    return mole, mole_ele

def all_bonds(atoms, tolerance=1.7):
    atoms_not_7 = []
    Bs = []
    Bs_ele = []
    for i, atom in enumerate(atoms):
        if atom.symbol not in ['H', 'F', 'Cl', 'Br']:
            atoms_not_7.append(i)
    for atomnum in atoms_not_7:
        mole, mole_ele = near_notH(atomnum, atoms, tolerance=tolerance)
        for dipole in mole:
            pre_dipole = [*dipole]
            dipole.sort()
            if dipole != pre_dipole:
                mole_ele.reverse()
            if dipole not in Bs:
                Bs.append(dipole)
                Bs_ele.append(mole_ele)
    return Bs, Bs_ele

In [3]:
def num_in_ring(num, ringsize):
    if num < ringsize:
        return num
    else:
        return num - ringsize

def classify_nodes_by_cycles(n, edges, atoms, simplerings=True):
    # Find all rings
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v)
        adj[v].append(u)
    
    all_cycles = []
    all_cycles_hide = []
    conjs = []
    visited_global = [False] * n
    #print('adj =', adj)
    
    def find_simple_cycles(start):
        visited_local = [False] * n
        stack = []
        
        def dfs(node, depth, min_node):
            visited_local[node] = True
            stack.append(node)
            
            for neighbor in adj[node]:
                if neighbor < min_node:
                    continue
                    
                if neighbor == start and depth >= 2:
                    
                    cycle = stack[:] + [start]
                    
                    min_idx = cycle.index(min(cycle))
                    normalized = tuple(cycle[min_idx:] + cycle[:min_idx])
                    if {*normalized} not in all_cycles_hide:
                        all_cycles.append(normalized)
                        all_cycles_hide.append({*normalized})
                elif not visited_local[neighbor] and neighbor >= min_node:
                    dfs(neighbor, depth + 1, min_node)
            
            stack.pop()
            visited_local[node] = False
        
        dfs(start, 0, start)
    
    # Find all rings
    for i in range(n):
        find_simple_cycles(i)
        # Avoid Repeating
        for j in range(n):
            if i in adj[j]:
                adj[j].remove(i)
    
    # Nodes to Cycles
    node_to_cycles = defaultdict(list)
    for cycle_idx, cycle in enumerate(all_cycles):
        #print(cycle_idx)
        for node in cycle[:-1]:
            if node not in node_to_cycles[node]:
                #print(cycle)
                node_to_cycles[node].append(cycle_idx)
    
    real_cycles = []
    
    if simplerings == True:
        for cycle in all_cycles:
            real_cycles.append(cycle)
    else:
        real_cycles = all_cycles
    
    for cycle in real_cycles:
        if len(cycle) <= 5:
            conjs.append(0)
        else:
            print(cycle)
            conjs.append(1)
            for i in range(len(cycle)):
                num1 = cycle[num_in_ring(0+i, len(cycle)-1)]
                num2 = cycle[num_in_ring(1+i, len(cycle)-1)]
                num3 = cycle[num_in_ring(2+i, len(cycle)-1)]
                num4 = cycle[num_in_ring(3+i, len(cycle)-1)]
                angle = atoms.get_dihedral(num1, num2, num3, num4)
                print(angle)
                if 1 <= angle <= 360-1:
                    conjs[-1] = 0
    return real_cycles, conjs

In [9]:
atoms = read('./Extra/2044_0.0.xyz')
bonds = all_bonds(atoms, tolerance=1.7)[0]

all_cycles, conjs = classify_nodes_by_cycles(len(atoms.get_chemical_symbols()), bonds, atoms)

for i, cycle in enumerate(all_cycles):
    print(f"Ring {i+1}: {cycle[:-1]}, {'' if conjs[i] == 1 else 'Not'} Conjugated.")

(0, 1, 2, 9, 10, 11, 8, 3, 0)
129.91621161471556
262.2965083394602
4.0895685276740785
1.06081894521433
345.5848515516346
125.85401464715561
252.84090105115743
346.8528713379688
129.91621161471556
(0, 1, 2, 9, 20, 21, 22, 23, 10, 11, 8, 3, 0)
129.91621161471556
83.34164400994706
178.12004786818463
359.4925000336833
1.410611317010177
359.06485788363517
177.91459200019526
182.68369024222892
345.5848515516346
125.85401464715561
252.84090105115743
346.8528713379688
129.91621161471556
(0, 1, 7, 6, 5, 4, 0)
4.356905541481663
4.145874082009289
0.855978280542857
345.79883135582133
22.00648369293762
343.1197680011215
4.356905541481663
(0, 1, 7, 17, 16, 15, 14, 6, 5, 4, 0)
186.08694797036938
178.45941261806811
359.5362401523693
0.26770697164873897
0.23572902822393527
177.46145885831726
182.84660903565612
345.79883135582133
22.00648369293762
343.1197680011215
186.08694797036938
(0, 3, 2, 1, 7, 6, 5, 4, 0)
346.8512583849154
129.94113967295567
262.26164217855643
4.145874082009289
0.855978280542857
3

In [72]:
print(all_cycles)

mix_rings = []
for cycle1 in all_cycles:
    for cycle2 in all_cycles:
        if cycle1 == cycle2:
            continue
        mix_ring = {*cycle1} | {*cycle2}
        mix_rings.append(mix_ring)
print()
print(mix_rings)
i = len(all_cycles) - 1

while i >= 0:
    print({*all_cycles[i]})
    if {*all_cycles[i]} in mix_rings:
        del all_cycles[i]
    i = i - 1

print()
print(all_cycles)

[(0, 1, 2, 3, 0), (0, 1, 2, 9, 10, 11, 8, 3, 0), (0, 1, 2, 9, 20, 21, 22, 23, 10, 11, 8, 3, 0), (0, 1, 7, 6, 5, 4, 0), (0, 1, 7, 17, 16, 15, 14, 6, 5, 4, 0), (0, 3, 2, 1, 7, 6, 5, 4, 0), (0, 3, 2, 1, 7, 17, 16, 15, 14, 6, 5, 4, 0), (0, 3, 8, 11, 10, 9, 2, 1, 7, 6, 5, 4, 0), (0, 3, 8, 11, 10, 9, 2, 1, 7, 17, 16, 15, 14, 6, 5, 4, 0), (0, 3, 8, 11, 10, 23, 22, 21, 20, 9, 2, 1, 7, 6, 5, 4, 0), (0, 3, 8, 11, 10, 23, 22, 21, 20, 9, 2, 1, 7, 17, 16, 15, 14, 6, 5, 4, 0), (2, 3, 8, 11, 10, 9, 2), (2, 3, 8, 11, 10, 23, 22, 21, 20, 9, 2), (6, 7, 17, 16, 15, 14, 6), (9, 10, 23, 22, 21, 20, 9)]

[{0, 1, 2, 3, 8, 9, 10, 11}, {0, 1, 2, 3, 8, 9, 10, 11, 20, 21, 22, 23}, {0, 1, 2, 3, 4, 5, 6, 7}, {0, 1, 2, 3, 4, 5, 6, 7, 14, 15, 16, 17}, {0, 1, 2, 3, 4, 5, 6, 7}, {0, 1, 2, 3, 4, 5, 6, 7, 14, 15, 16, 17}, {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11}, {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 14, 15, 16, 17}, {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 20, 21, 22, 23}, {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 14, 15, 16

In [10]:
def all_smallest_rings(atoms, tolerance=1.7):
    bonds = all_bonds(atoms, tolerance=1.7)[0]
    all_cycles, conjs = classify_nodes_by_cycles(len(atoms.get_chemical_symbols()), bonds, atoms)

    mix_rings = []
    for cycle1 in all_cycles:
        for cycle2 in all_cycles:
            if cycle1 == cycle2:
                continue
            mix_ring = {*cycle1} | {*cycle2}
            mix_rings.append(mix_ring)
    i = len(all_cycles) - 1

    while i >= 0:
        if {*all_cycles[i]} in mix_rings:
            del all_cycles[i]
        i = i - 1
    
    return all_cycles

atoms = read('./Extra/2044_0.0.xyz')
rings = all_smallest_rings(atoms, tolerance=1.7)
print(rings)

(0, 1, 2, 9, 10, 11, 8, 3, 0)
129.91621161471556
262.2965083394602
4.0895685276740785
1.06081894521433
345.5848515516346
125.85401464715561
252.84090105115743
346.8528713379688
129.91621161471556
(0, 1, 2, 9, 20, 21, 22, 23, 10, 11, 8, 3, 0)
129.91621161471556
83.34164400994706
178.12004786818463
359.4925000336833
1.410611317010177
359.06485788363517
177.91459200019526
182.68369024222892
345.5848515516346
125.85401464715561
252.84090105115743
346.8528713379688
129.91621161471556
(0, 1, 7, 6, 5, 4, 0)
4.356905541481663
4.145874082009289
0.855978280542857
345.79883135582133
22.00648369293762
343.1197680011215
4.356905541481663
(0, 1, 7, 17, 16, 15, 14, 6, 5, 4, 0)
186.08694797036938
178.45941261806811
359.5362401523693
0.26770697164873897
0.23572902822393527
177.46145885831726
182.84660903565612
345.79883135582133
22.00648369293762
343.1197680011215
186.08694797036938
(0, 3, 2, 1, 7, 6, 5, 4, 0)
346.8512583849154
129.94113967295567
262.26164217855643
4.145874082009289
0.855978280542857
3

In [73]:
def cond_append(items, ls):
    for item in items:
        if item not in ls:
            ls.append(item)
    return ls

print(all_cycles)

route = [28,26,15,7,1,2,9,22,27,32]

atoms = [1,2]

max_ring_on_minroute = []

for cycle in all_cycles:
    assemb = {*cycle} & {*route}
    if len(assemb) >= 2:
        max_ring_on_minroute = cond_append(cycle, max_ring_on_minroute)

print()
print(max_ring_on_minroute)
print()

max_ring_on_bond = [*atoms]
current_max_ring_on_bond = []
while 1:
    for cycle in all_cycles:
        index = 0
        for atom in max_ring_on_bond:
            if atom in cycle:
                index += 1
        if index >= 2:
            max_ring_on_bond = cond_append(cycle, max_ring_on_bond)
    if current_max_ring_on_bond == max_ring_on_bond:
        break
    else:
        current_max_ring_on_bond = [*max_ring_on_bond]

print(max_ring_on_bond)

print({*max_ring_on_minroute} & {*max_ring_on_bond})

[(0, 1, 2, 3, 0), (0, 1, 7, 6, 5, 4, 0), (2, 3, 8, 11, 10, 9, 2), (6, 7, 17, 16, 15, 14, 6), (9, 10, 23, 22, 21, 20, 9)]

[0, 1, 2, 3, 7, 6, 5, 4, 8, 11, 10, 9, 17, 16, 15, 14, 23, 22, 21, 20]

[1, 2, 0, 3, 7, 6, 5, 4, 8, 11, 10, 9, 17, 16, 15, 14, 23, 22, 21, 20]
{0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 14, 15, 16, 17, 20, 21, 22, 23}


In [6]:
#csv = pd.read_csv('./generalization/final_isbroken_test_transfer.csv')
#csv = pd.read_csv('candidate_bonds.csv')
csv = pd.read_csv('Force_candidate_transfer.csv')

Molnums = csv['Molecule'].tolist()
Broken_bonds1 = csv['Atom 1'].tolist()
Broken_bonds2 = csv['Atom 2'].tolist()

molnum = '---'

w = open('ring.csv', 'w')
w.write('Molecule,InRing_B,InRing_1,InRing_2\n')

for i, mol in enumerate(Molnums):
    
    if Molnums[i] != molnum:
        molnum = Molnums[i]
        atoms = read('./Extra/ir/xyz/{0}_0.0.xyz'.format(mol))
        rings = all_smallest_rings(atoms, tolerance=1.7)
    
    bond = [Broken_bonds1[i], Broken_bonds2[i]]
    atom1 = Broken_bonds1[i]
    atom2 = Broken_bonds2[i]
    ringsize_b = 99999
    ringsize_1 = 99999
    ringsize_2 = 99999
    
    for ring in rings:
        andring = {*bond} & {*ring}
        if len(andring) >= 2 and len(ring) - 1 < ringsize_b:
            ringsize_b = len(ring) - 1
        if atom1 in ring and len(ring) - 1 < ringsize_1:
            ringsize_1 = len(ring) - 1
        if atom2 in ring and len(ring) - 1 < ringsize_2:
            ringsize_2 = len(ring) - 1
    
    if ringsize_b == 99999:
        ringsize_b = 0
    elif 7 <= ringsize_b < 99:
        ringsize_b = 7
    if ringsize_1 == 99999:
        ringsize_1 = 0
    elif 7 <= ringsize_1 < 99:
        ringsize_1 = 7
    if ringsize_2 == 99999:
        ringsize_2 = 0
    elif 7 <= ringsize_2 < 99:
        ringsize_2 = 7

    w.write('{0},{1},{2},{3}\n'.format(mol, ringsize_b, ringsize_1, ringsize_2))
    print('Molecule {0}:'.format(mol))
    print('Bond:',bond)
    print(ringsize_b, ringsize_1, ringsize_2)
    print()

w.close()

(0, 1, 2, 6, 5, 4, 3, 0)
280.93654456119185
94.36673822077998
353.07288640865164
274.85090958608373
87.72976161700343
348.4915680641586
7.03023150665134
280.93654456119185
(0, 1, 2, 28, 4, 3, 0)
35.38759657809196
288.15358623597155
67.88086592944497
333.43108165117195
348.4915680641586
7.03023150665134
35.38759657809196
(2, 6, 5, 4, 28, 2)
353.07288640865164
34.45453289101066
308.9738537730284
46.344696247209676
337.5408145900393
353.07288640865164
Molecule 4001:
Bond: [4, 5]
5 5 5

Molecule 4001:
Bond: [3, 4]
6 6 5

Molecule 4001:
Bond: [0, 3]
6 6 6

Molecule 4001:
Bond: [5, 16]
0 5 0

Molecule 4001:
Bond: [7, 0]
0 0 6

Molecule 4001:
Bond: [18, 19]
0 0 0

Molecule 4001:
Bond: [9, 8]
0 0 0

Molecule 4001:
Bond: [8, 7]
0 0 0

Molecule 4001:
Bond: [16, 18]
0 0 0

(0, 1, 2, 6, 5, 4, 3, 0)
279.5325051940487
86.34129139849563
5.17880708371731
267.7010053564809
76.63354171131967
359.36418076853306
1.6788910469806895
279.5325051940487
(0, 1, 2, 12, 4, 3, 0)
34.20309076039134
287.062586105329

In [7]:
#csv = pd.read_csv('./generalization/final_isbroken_test_orig.csv')
#csv = pd.read_csv('./generalization/final_isbroken_test_complete_transfer.csv')
#csv = pd.read_csv('candidate_bonds.csv')
csv = pd.read_csv('Force_candidate_transfer.csv')

Molnums = csv['Molecule'].tolist()
Broken_bonds1 = csv['Atom 1'].tolist()
Broken_bonds2 = csv['Atom 2'].tolist()

molnum = '---'

w = open('ring.csv', 'w')
w.write('Molecule,InRing_1,InRing_2\n')

for i, mol in enumerate(Molnums):
    
    if Molnums[i] != molnum:
        molnum = Molnums[i]
        atoms = read('./Extra/ir/xyz/{0}_0.0.xyz'.format(mol))
        rings = all_smallest_rings(atoms, tolerance=1.7)
    
    bond = [Broken_bonds1[i], Broken_bonds2[i]]
    atom1 = Broken_bonds1[i]
    atom2 = Broken_bonds2[i]
    ringsize_1 = 99999
    ringsize_2 = 99999
    
    for ring in rings:
        andring = {*bond} & {*ring}
        if atom1 in ring and len(ring) - 1 < ringsize_1:
            ringsize_1 = len(ring) - 1
        if atom2 in ring and len(ring) - 1 < ringsize_2:
            ringsize_2 = len(ring) - 1

    if ringsize_1 == 99999:
        ringsize_1 = 0
    elif 7 <= ringsize_1 < 99:
        ringsize_1 = 7
    if ringsize_2 == 99999:
        ringsize_2 = 0
    elif 7 <= ringsize_2 < 99:
        ringsize_2 = 7

    w.write('{0},{1},{2}\n'.format(mol, ringsize_1, ringsize_2))
    print('Molecule {0}:'.format(mol))
    print('Bond:',bond)
    print(ringsize_1, ringsize_2)
    print()

w.close()

(0, 1, 2, 6, 5, 4, 3, 0)
280.93654456119185
94.36673822077998
353.07288640865164
274.85090958608373
87.72976161700343
348.4915680641586
7.03023150665134
280.93654456119185
(0, 1, 2, 28, 4, 3, 0)
35.38759657809196
288.15358623597155
67.88086592944497
333.43108165117195
348.4915680641586
7.03023150665134
35.38759657809196
(2, 6, 5, 4, 28, 2)
353.07288640865164
34.45453289101066
308.9738537730284
46.344696247209676
337.5408145900393
353.07288640865164
Molecule 4001:
Bond: [4, 5]
5 5

Molecule 4001:
Bond: [3, 4]
6 5

Molecule 4001:
Bond: [0, 3]
6 6

Molecule 4001:
Bond: [5, 16]
5 0

Molecule 4001:
Bond: [7, 0]
0 6

Molecule 4001:
Bond: [18, 19]
0 0

Molecule 4001:
Bond: [9, 8]
0 0

Molecule 4001:
Bond: [8, 7]
0 0

Molecule 4001:
Bond: [16, 18]
0 0

(0, 1, 2, 6, 5, 4, 3, 0)
279.5325051940487
86.34129139849563
5.17880708371731
267.7010053564809
76.63354171131967
359.36418076853306
1.6788910469806895
279.5325051940487
(0, 1, 2, 12, 4, 3, 0)
34.20309076039134
287.0625861053297
73.0596358322036

In [4]:
# Find Routes

f = open('minroute.csv', 'r')

minroutes = {}

for line in f:
    if 'Molecule' in line:
        continue
    a = re.split(',', line)
    mol = a[0]
    r = re.search(r'\[.*?\]', line)
    minroutes[mol] = eval(r.group(0))
    print(mol, r.group(0))

1 [31, 30, 24, 17, 16, 1, 0, 2, 4, 11, 15, 20, 21, 27, 35, 36]
2 [31, 30, 24, 17, 16, 1, 0, 2, 4, 11, 15, 20, 21, 27, 35, 36]
3 [31, 30, 24, 17, 16, 1, 0, 2, 4, 11, 15, 20, 21, 27, 35, 36]
4 [30, 29, 23, 16, 15, 1, 0, 2, 4, 11, 14, 19, 20, 26, 34, 35]
5 [29, 28, 22, 15, 14, 1, 0, 2, 4, 10, 13, 18, 19, 25, 33, 34]
6 [29, 28, 22, 15, 14, 1, 0, 2, 4, 10, 13, 18, 19, 25, 33, 34]
7 [36, 33, 28, 24, 26, 30, 35, 40]
8 [23, 22, 20, 4, 2, 1, 9, 34, 35, 12, 11, 18, 17, 21, 27, 28]
9 [23, 22, 20, 4, 2, 1, 9, 34, 35, 12, 11, 18, 17, 21, 27, 28]
10 [44, 42, 36, 34, 32, 38, 43, 49]
11 [17, 16, 13, 4, 0, 1, 10, 15, 23, 24]
12 [8, 7, 4, 3, 1, 0, 13, 16, 17]
13 [34, 32, 19, 18, 7, 8, 1, 0, 2, 12, 14, 28, 27, 33, 38]
14 [30, 28, 15, 14, 6, 7, 1, 2, 9, 10, 24, 23, 29, 34]
15 [48, 47, 44, 41, 40, 38, 25, 24, 5, 4, 0, 2, 9, 10, 34, 33, 39, 52, 53, 57, 60, 61]
16 [28, 27, 22, 18, 20, 24, 26, 32]
17 [27, 26, 21, 17, 19, 23, 25, 31]
18 [32, 31, 29, 25, 27, 38, 40, 41]
19 [33, 32, 30, 26, 28, 39, 41, 42]
20 [1

In [6]:
# Old Verision

In [26]:
def cond_append(items, ls):
    for item in items:
        if item not in ls:
            ls.append(item)
    return ls

def maximum_cycle_and(route, bond, all_cycles, specialspiro=[]):
    max_cycle_on_minroute = []
    
    if specialspiro != []:
        specialspiroatom = {*specialspiro} & {*route}
        specialspiroatom = [*specialspiroatom]

    for cycle in all_cycles:
        assemb0 = {*cycle} & {*route}
        if len(assemb0) >= 2:
            assemb = {*cycle} & {*route}
            max_cycle_on_minroute = cond_append(cycle, max_cycle_on_minroute)
        elif len(assemb0) == 1 and specialspiro != []:
            if [*assemb0][0] in specialspiroatom:
                assemb = {*cycle} & {*route}
                max_cycle_on_minroute = cond_append(cycle, max_cycle_on_minroute)

    max_cycle_on_bond = [*bond]
    current_max_cycle_on_bond = []
    j = 0
    while 1:
        for cycle in all_cycles:
            index = 0
            for atom in max_cycle_on_bond:
                if atom in cycle:
                    index += 1
            if index >= 2 and j == 0:
                max_cycle_on_bond = cond_append(cycle, max_cycle_on_bond)
            elif index >= 1 and j > 0:
                max_cycle_on_bond = cond_append(cycle, max_cycle_on_bond)
        print(current_max_cycle_on_bond, [*max_cycle_on_bond])
        if current_max_cycle_on_bond == max_cycle_on_bond:
            break
        elif len(max_cycle_on_bond) < 3:
            return []
        else:
            current_max_cycle_on_bond = [*max_cycle_on_bond]
        j = j + 1

    result = {*max_cycle_on_minroute} & {*max_cycle_on_bond}
    return [*result]

def spiropointsfunc(rings, func='delta', limit=10):
    bridgepoints = []
    spiropoints = []
    for cycle1 in rings:
        for cycle2 in rings:
            if cycle1 == cycle2:
                continue
            and_ring = {*cycle1} & {*cycle2}
            if len(and_ring) == 1:
                ele = [*and_ring][0]
                if ele not in spiropoints:
                    spiropoints.append([*and_ring][0])
            elif len(and_ring) >= 2:
                if len(cycle1) <= limit and len(cycle2) <= limit:
                    eles = [*and_ring]
                    for ele in eles:
                        if ele not in spiropoints:
                            bridgepoints.append([*and_ring][0])
    if func == 'delta':
        delta_spiropoints = {*spiropoints} - {*bridgepoints}
        return [*delta_spiropoints]
    elif func == 'all':
        return spiropoints

def spirocountfunc(all_cycles, atoms, tolerance=1.7):
    all_spiro_points = []
    all_spiro_points_num = []
    for cycle1 in all_cycles:
        for cycle2 in all_cycles:
            if cycle1 == cycle2:
                continue
            cycle1_set = {*cycle1}
            cycle2_set = {*cycle2}
            and_set = {*cycle1} & {*cycle2}
            if len(and_set) == 1:
                spiro_point_num = [*and_set][0]
                if spiro_point_num in all_spiro_points_num:
                    continue
                rest_cycle1 = [*cycle1]
                rest_cycle2 = [*cycle2]
                print(cycle1, cycle2, spiro_point_num)
                rest_cycle1.remove(spiro_point_num)
                rest_cycle2.remove(spiro_point_num)
                
                mole, mole_ele = near_notH(spiro_point_num, atoms, tolerance=tolerance)
                neigh_atoms = []
                for mol in mole:
                    for atom in mol:
                        if atom not in neigh_atoms:
                            neigh_atoms.append(atom)
                neigh_atoms.remove(spiro_point_num)
                
                #print(neigh_atoms)
                
                neigh_cycle1 = []
                neigh_cycle2 = []
                
                for atom in neigh_atoms:
                    if atom in rest_cycle1:
                        neigh_cycle1.append(atom)
                    elif atom in rest_cycle2:
                        neigh_cycle2.append(atom)
                
                all_spiro_points.append([spiro_point_num, neigh_cycle1, neigh_cycle2])
                all_spiro_points_num.append(spiro_point_num)
    return all_spiro_points

def all_smallest_rings(atoms, tolerance=1.7):
    bonds = all_bonds(atoms, tolerance=1.7)[0]
    all_cycles, conjs = classify_nodes_by_cycles(len(atoms.get_chemical_symbols()), bonds, atoms)

    mix_rings = []
    for cycle1 in all_cycles:
        for cycle2 in all_cycles:
            if cycle1 == cycle2:
                continue
            mix_ring = {*cycle1} | {*cycle2}
            mix_rings.append(mix_ring)
    i = len(all_cycles) - 1

    while i >= 0:
        if {*all_cycles[i]} in mix_rings:
            del all_cycles[i]
        i = i - 1
    
    return all_cycles

csv = pd.read_csv('./generalization/final_isbroken_test_transfer.csv')
#csv = pd.read_csv('Force_candidate_transfer.csv')

Molnums = csv['Molecule'].tolist()
Broken_bonds1 = csv['Atom 1'].tolist()
Broken_bonds2 = csv['Atom 2'].tolist()

molnum = '---'

w = open('ratio.csv', 'w')
w.write('Molecule,MaxRing,Ratio\n')

for i, mol in enumerate(Molnums):
    if mol != '4049':
        continue
    print('Molecule {0}:'.format(mol))
    
    if Molnums[i] != molnum:
        molnum = Molnums[i]
        atoms = read('./Extra/ir/xyz/{0}_0.0.xyz'.format(mol))
        rings = all_smallest_rings(atoms, tolerance=1.7)
        spiropoints = spiropointsfunc(rings)
        specialspiropoints = spiropointsfunc(rings, func='all')
        route = minroutes[molnum]
    print(molnum, mol)
    
    print('Rings:', rings)
    print('Route:', route)
    print('Spiro-Points:', spiropoints)
    
    bond = [Broken_bonds1[i], Broken_bonds2[i]]
    print('Bond:',bond)
    maximum_cycle = maximum_cycle_and(route, bond, rings, specialspiro=specialspiropoints)
    maximum_cycle_in_route = {*maximum_cycle} & {*route}
    print(maximum_cycle, route)
    print(len(maximum_cycle_in_route), len(maximum_cycle))
    print(maximum_cycle, maximum_cycle_in_route)
    
    spiropoints2 = {*specialspiropoints} & maximum_cycle_in_route
    
    if maximum_cycle != 0 and maximum_cycle_in_route != 0:
        maximum_cycle_routes = allroutes('./Extra/ir/xyz/{0}_0.0'.format(mol), atomnums=bond, tolerance=2.000,\
                                        form='XYZ', must=maximum_cycle_in_route, rangelimit=maximum_cycle, spiropoints=spiropoints2, verbose=False)
        #print(bond, maximum_ring_in_route, maximum_ring, spiropoints)
        maximum_cycle_min_routes = 99999
        print([len(route0) for route0 in maximum_cycle_routes])
        for route0 in maximum_cycle_routes:
            if len(route0) < maximum_cycle_min_routes and len(route0) != 2:
                maximum_cycle_min_routes = len({*route0})
    
    if maximum_cycle_min_routes == 99999:
        maximum_cycle_min_routes = 0
        
    print(maximum_cycle_in_route, maximum_cycle_min_routes)

    # Count spiro_points (format example: [15, [3,6], [16,21]])  
    all_spiro_points = spirocountfunc(rings, atoms, tolerance=1.7)

    print(all_spiro_points)

    spiro_index = 0
    for i in range(len(all_spiro_points)):
        if all_spiro_points[i][0] in maximum_cycle:
            spirofrag1 = {*all_spiro_points[i][1]}
            spirofrag2 = {*all_spiro_points[i][2]}
            maximum_cycle_set = {*maximum_cycle}
            if spirofrag1 & maximum_cycle_set != {} and spirofrag2 & maximum_cycle_set != {}:
                spiro_index = spiro_index + 1


    len_maximum_cycle = len(maximum_cycle) + spiro_index
    print(len_maximum_cycle)

    if len(maximum_cycle_in_route) == 0:
        ratio_intro = 0
    else:
        #ratio_intro = (len(maximum_cycle_in_route) - 1) / len(maximum_cycle)
        #ratio_intro = (len(maximum_cycle_in_route) - 1) / maximum_cycle_min_routes
        ratio_intro = (len(maximum_cycle_in_route) - 1) / len_maximum_cycle
    print('Cycle:', maximum_cycle, len(maximum_cycle))
    
    #print('Intra Ratio: {0} / {1} = {2}'.format(len(maximum_cycle_in_route) - 1, len(maximum_cycle)),ratio_intro)
    print('Intra Ratio: {0} / {1} = {2}'.format(len(maximum_cycle_in_route) - 1, len_maximum_cycle, ratio_intro))

    w.write('{0},{1},{2}\n'.format(mol, maximum_cycle_min_routes, ratio_intro))
    print()
    break

w.close()

Molecule 4049:
(0, 1, 3, 4, 6, 8, 0)
358.9202520455199
359.208955121272
1.224811731133205
0.19352620853014166
357.96273884601
2.464075671438221
358.9202520455199
(0, 1, 3, 4, 6, 8, 18, 14, 13, 19, 0)
358.9202520455199
359.208955121272
1.224811731133205
182.54382565843562
231.80897462966539
305.0400274142527
1.551045008919501
52.60060020031976
132.11930418479423
176.5940494849069
358.9202520455199
(0, 1, 3, 4, 6, 8, 18, 14, 15, 17, 10, 11, 13, 19, 0)
358.9202520455199
359.208955121272
1.224811731133205
182.54382565843562
231.80897462966539
126.96199990152417
178.02721157404085
359.2158867748549
0.44372224958432815
0.6085521544817322
180.09670900363466
231.27061945258686
132.11930418479423
176.5940494849069
358.9202520455199
(0, 1, 3, 4, 6, 8, 18, 21, 20, 19, 0)
358.9202520455199
359.208955121272
1.224811731133205
182.54382565843562
119.07132412325483
52.236146598095274
6.325362980235237
298.749301754881
244.18824978453475
176.5940494849069
358.9202520455199
(0, 1, 3, 4, 6, 8, 18, 21, 26

In [71]:
def cond_append(items, ls):
    for item in items:
        if item not in ls:
            ls.append(item)
    return ls

def cycles_size_rank(cycles):
    cycle_size = []
    ranked_cycles = []
    cond_append([len(cycle) for cycle in cycles], cycle_size)
    cycle_size.sort()
    for size in cycle_size:
        for cycle in cycles:
            if len(cycle) == size:
                ranked_cycles.append(cycle)
    return ranked_cycles

def maximum_cycle_and(route, scissile_bond, all_cycles, specialspiro=[]):
    #
    max_cycle_on_minroute = []
    route_overlap = {}
    route_overlap_iter = {}
    for num in route:
        route_overlap_iter[num] = 0
    
    #if specialspiro != []:
    #    specialspiroatom = {*specialspiro} & {*route}
    #    specialspiroatom = [*specialspiroatom]
    #
    #print('Special Spiro Atoms:', specialspiro)

    #for cycle in all_cycles:
    #    assemb0 = {*cycle} & {*route}
    #    if len(assemb0) >= 2:
    #        assemb = {*cycle} & {*route}
    #        max_cycle_on_minroute = cond_append(cycle, max_cycle_on_minroute)
    #    elif len(assemb0) == 1 and specialspiro != []:
    #        if [*assemb0][0] in specialspiroatom:
    #            assemb = {*cycle} & {*route}
    #            max_cycle_on_minroute = cond_append(cycle, max_cycle_on_minroute)
    
    for i in range(len(route)-1):
        bond = [route[i], route[i+1]]
        
        smallest_rings = []
        smallest_cycle_size = 99999
        for cycle in all_cycles:
            assemb0 = {*cycle} & {*bond}
            if len(assemb0) == 2:
                if len(cycle) < smallest_cycle_size:
                    smallest_cycle_size = len(cycle)
        #print(smallest_cycle_size)
        for cycle in all_cycles:
            assemb0 = {*cycle} & {*bond}
            if len(assemb0) == 2:
                if len(cycle) == smallest_cycle_size:
                    max_cycle_on_minroute = cond_append(cycle, max_cycle_on_minroute)
    
    print('Max Cycle on Minroute:', max_cycle_on_minroute)
    
    #
    max_cycle_on_bond = [*scissile_bond]
    current_max_cycle_on_bond = []
    j = 0
    
    while 1:
        route_overlap_box = []
        overlap = []
        ring_correlated_to_overlap = []
        
        for cycle in all_cycles:
            for num in route:
                route_overlap[num] = 0
            index = 0
            for atom in max_cycle_on_bond:
                if atom in cycle:
                    index += 1
            if (index == 2 and j == 0):
                route_set = {*route}
                cycle_set = {*cycle}
                and_set = route_set & cycle_set
                for num in [*and_set]:
                    route_overlap[num] = 1
                route_overlap_box.append([*list(route_overlap.values())])#
                overlap.append(len(and_set))
                ring_correlated_to_overlap.append(cycle)
            elif (index == 1 and j > 0):
                route_set = {*route}
                cycle_set = {*cycle}
                and_set = route_set & cycle_set
                for num in [*and_set]:
                    route_overlap[num] = 1
                route_overlap_box.append([*list(route_overlap.values())])
                overlap.append(len(and_set))
                ring_correlated_to_overlap.append(cycle)
            elif (index == 2 and j > 0):
                route_set = {*route}
                cycle_set = {*cycle}
                and_set = route_set & cycle_set
                if {*scissile_bond} != and_set:
                    for num in [*and_set]:
                        route_overlap[num] = 1
                    route_overlap_box.append([*list(route_overlap.values())])
                    overlap.append(len(and_set))
                    ring_correlated_to_overlap.append(cycle)
                    #max_cycle_on_bond = cond_append(cycle, max_cycle_on_bond)
                    #break
        
        #print(route_overlap_iter)
        print('Correlated to:',ring_correlated_to_overlap)
        #if j == 0 or (overlap != [] and j > 0):
        if overlap != []:
            # 1a. With New Overlap
            overlap_judgement = []
            pre_overlap = list(route_overlap_iter.values())
            pre_overlap_tests = []
            for route_overlap0 in route_overlap_box:
                pre_overlap_test = [pre_overlap[i] | route_overlap0[i] for i in range(len(pre_overlap))]
                print(pre_overlap_test, pre_overlap, route_overlap0)
                if pre_overlap_test == pre_overlap:
                    overlap_judgement.append(0)
                    pre_overlap_tests.append([])
                else:
                    overlap_judgement.append(1)
                    pre_overlap_tests.append(pre_overlap_test)
                    
            # 1b. Overlap Most
            best_overlap_cycle_size = -99999
            for k in range(len(overlap)):
                if overlap[k] > best_overlap_cycle_size and overlap_judgement[k] == 1:
                    best_overlap_cycle_size = overlap[k]
                    best_overlap_no = k
            best_overlap_box = []
            route_overlap_box_2 = []
            for k in range(len(overlap)):
                if overlap[k] == best_overlap_cycle_size and overlap_judgement[k] == 1:
                    best_overlap_box.append(ring_correlated_to_overlap[k])
                    route_overlap_box_2.append(pre_overlap_tests[k])
            print('Candidate Rings:', best_overlap_box)
            
            # 2. Smallest ring
            print(best_overlap_box, route_overlap_box_2)
            if best_overlap_box == []:
                print(0)
                pass
            elif len(best_overlap_box) == 1:
                max_cycle_on_bond = cond_append(best_overlap_box[0], max_cycle_on_bond)
                for ii, num in enumerate(route):
                    route_overlap_iter[num] = route_overlap_box_2[0][ii]
                print(1)
            else:
                print(2)
                smallest_cycle_size = 99999
                for ring in best_overlap_box:
                    if len(ring) < smallest_cycle_size:
                        smallest_cycle_size = len(ring)
                for ii, ring in enumerate(best_overlap_box):
                    if len(ring) == smallest_cycle_size:
                        max_cycle_on_bond = cond_append(ring, max_cycle_on_bond)
                        #
                        for ij, num in enumerate(route):
                            route_overlap_iter[num] = route_overlap_box_2[ii][ij]
                        break

        #print(current_max_cycle_on_bond, [*max_cycle_on_bond])
        if current_max_cycle_on_bond == max_cycle_on_bond:
            break
        elif len(max_cycle_on_bond) < 3:
            return []
        else:
            current_max_cycle_on_bond = [*max_cycle_on_bond]
        j = j + 1
    print('Max Cycle on Scissile Bond:', max_cycle_on_bond)

    result = {*max_cycle_on_minroute} & {*max_cycle_on_bond}
    return [*result]

def spiropointsfunc(rings, func='delta', limit=10):
    fusepoints = []
    spiropoints = []
    for cycle1 in rings:
        for cycle2 in rings:
            if cycle1 == cycle2:
                continue
            and_ring = {*cycle1} & {*cycle2}
            if len(and_ring) == 1:
                ele = [*and_ring][0]
                if ele not in spiropoints:
                    spiropoints.append([*and_ring][0])
            elif len(and_ring) == 2:
                if len(cycle1) <= limit and len(cycle2) <= limit:
                    eles = [*and_ring]
                    for ele in eles:
                        if ele not in spiropoints:
                            fusepoints.append([*and_ring][0])
    if func == 'delta':
        delta_spiropoints = {*spiropoints} - {*fusepoints}
        return [*delta_spiropoints]
    elif func == 'all':
        return spiropoints

def spirocountfunc(all_cycles, atoms, tolerance=1.7, grid_shape_detect=False, limit_to = 'ALL'):
    all_spiro_points = []
    all_spiro_points_num = []
    all_spiro_points_occurrence = {}
    if limit_to == 'ALL':
        pass
    elif type(limit_to) == list:
        limitto_set = {*limit_to}
        rectified_all_cycles = []
        for cycle in all_cycles:
            and_result = limitto_set | {*cycle}
            if and_result == limitto_set:
                rectified_all_cycles.append(cycle)
        all_cycles = rectified_all_cycles
    else:
        raise TypeError("The index 'limit_to' can only be set as 'ALL' or list of atoms.")
    for cycle1 in all_cycles:
        for cycle2 in all_cycles:
            if cycle1 == cycle2:
                continue
            cycle1_set = {*cycle1}
            cycle2_set = {*cycle2}
            and_set = {*cycle1} & {*cycle2}
            if len(and_set) == 1:
                spiro_point_num = [*and_set][0]
                if spiro_point_num in all_spiro_points_num:
                    all_spiro_points_occurrence[spiro_point_num] += 1
                    continue
                rest_cycle1 = [*cycle1]
                rest_cycle2 = [*cycle2]
                print(cycle1, cycle2, spiro_point_num)
                rest_cycle1.remove(spiro_point_num)
                rest_cycle2.remove(spiro_point_num)
                
                mole, mole_ele = near_notH(spiro_point_num, atoms, tolerance=tolerance)
                neigh_atoms = []
                for mol in mole:
                    for atom in mol:
                        if atom not in neigh_atoms:
                            neigh_atoms.append(atom)
                neigh_atoms.remove(spiro_point_num)
                
                #print(neigh_atoms)
                
                neigh_cycle1 = []
                neigh_cycle2 = []
                
                for atom in neigh_atoms:
                    if atom in rest_cycle1:
                        neigh_cycle1.append(atom)
                    elif atom in rest_cycle2:
                        neigh_cycle2.append(atom)
                
                if [spiro_point_num, neigh_cycle1, neigh_cycle2] not in all_spiro_points:
                    all_spiro_points.append([spiro_point_num, neigh_cycle1, neigh_cycle2])
                    all_spiro_points_num.append(spiro_point_num)
                try:
                    all_spiro_points_occurrence[spiro_point_num] = 1
                except:
                    all_spiro_points_occurrence[spiro_point_num] += 1
    if grid_shape_detect == True:
        length0 = len(all_spiro_points)
        for i in range(length0):
            spiro_point = all_spiro_points_num[length0-i-1]
            if all_spiro_points_occurrence[spiro_point] >= 3:
                del all_spiro_points[length0-i-1]
    return all_spiro_points

def all_smallest_rings(atoms, tolerance=1.7):
    bonds = all_bonds(atoms, tolerance=1.7)[0]
    all_cycles, conjs = classify_nodes_by_cycles(len(atoms.get_chemical_symbols()), bonds, atoms)

    mix_rings = []
    for cycle1 in all_cycles:
        for cycle2 in all_cycles:
            if cycle1 == cycle2:
                continue
            mix_ring = {*cycle1} | {*cycle2}
            mix_rings.append(mix_ring)
    i = len(all_cycles) - 1

    while i >= 0:
        if {*all_cycles[i]} in mix_rings:
            del all_cycles[i]
        i = i - 1
    
    return all_cycles

csv = pd.read_csv('./generalization_planB/final_isbroken_test_transfer.csv')
#csv = pd.read_csv('Force_candidate_transfer.csv')

Molnums = csv['Molecule'].tolist()
Broken_bonds1 = csv['Atom 1'].tolist()
Broken_bonds2 = csv['Atom 2'].tolist()

molnum = '---'

w = open('ratio2.csv', 'w')
w.write('Molecule,MaxRing,Ratio\n')

oooo = 0
for i, mol in enumerate(Molnums):
    if mol != '4117':
    #    oooo = 1
    #if oooo == 0:
        continue
    print('Molecule {0}:'.format(mol))
    
    if Molnums[i] != molnum:
        molnum = Molnums[i]
        atoms = read('./Extra/ir/xyz/{0}_0.0.xyz'.format(mol))
        rings = all_smallest_rings(atoms, tolerance=1.7)
        spiropoints = spiropointsfunc(rings)
        specialspiropoints = spiropointsfunc(rings, func='all')
        route = minroutes[molnum]
    print(molnum, mol)
    rings = cycles_size_rank(rings)
    
    print('Rings:', rings)
    print('Route:', route)
    print('Spiro-Points:', spiropoints)
    
    bond = [Broken_bonds1[i], Broken_bonds2[i]]
    print('Bond:',bond)
    maximum_cycle = maximum_cycle_and(route, bond, rings, specialspiro=specialspiropoints)
    maximum_cycle_in_route = {*maximum_cycle} & {*route}
    print(maximum_cycle, route)
    print(len(maximum_cycle_in_route), len(maximum_cycle))
    print(maximum_cycle, maximum_cycle_in_route)
    
    spiropoints2 = {*specialspiropoints} & maximum_cycle_in_route
    
    if maximum_cycle != 0 and maximum_cycle_in_route != 0:
        maximum_cycle_routes = allroutes('./Extra/ir/xyz/{0}_0.0'.format(mol), atomnums=bond, tolerance=2.000,\
                                         form='XYZ', must=maximum_cycle_in_route, rangelimit=maximum_cycle, spiropoints=spiropoints2, verbose=False)
        maximum_cycle_min_routes = 99999
        print([len(route0) for route0 in maximum_cycle_routes])
        for route0 in maximum_cycle_routes:
            if len(route0) < maximum_cycle_min_routes and len(route0) != 2:
                maximum_cycle_min_routes = len({*route0})
    
    if maximum_cycle_min_routes == 99999:
        maximum_cycle_min_routes = 0
        
    print(maximum_cycle_in_route, maximum_cycle_min_routes)

    # Count spiro_points (format example: [15, [3,6], [16,21]])  
    all_spiro_points = spirocountfunc(rings, atoms, tolerance=1.7, limit_to=maximum_cycle)

    spiro_index = 0
    for i in range(len(all_spiro_points)):
        if all_spiro_points[i][0] in maximum_cycle_in_route:
            spirofrag1 = {*all_spiro_points[i][1]}
            spirofrag2 = {*all_spiro_points[i][2]}
            maximum_cycle_set = {*maximum_cycle}
            if spirofrag1 & maximum_cycle_set != {} and spirofrag2 & maximum_cycle_set != {}:
                spiro_index = spiro_index + 1

    len_maximum_cycle = len(maximum_cycle) + spiro_index
    print(len_maximum_cycle)

    if len(maximum_cycle_in_route) == 0:
        ratio_intro = 0
    else:
        #ratio_intro = (len(maximum_cycle_in_route) - 1) / len(maximum_cycle)
        #ratio_intro = (len(maximum_cycle_in_route) - 1) / maximum_cycle_min_routes
        ratio_intro = (len(maximum_cycle_in_route) - 1) / len_maximum_cycle
    print('Cycle:', maximum_cycle, len(maximum_cycle))
    
    #print('Intra Ratio: {0} / {1} = {2}'.format(len(maximum_cycle_in_route) - 1, len(maximum_cycle)),ratio_intro)
    print('Intra Ratio: {0} / {1} = {2}'.format(len(maximum_cycle_in_route) - 1, len_maximum_cycle, ratio_intro))

    w.write('{0},{1},{2}\n'.format(mol, len_maximum_cycle, ratio_intro))
    print()
    #break

w.close()

Molecule 4117:
(0, 1, 2, 4, 3, 0)
341.56001532937853
39.933322128517524
311.42580055332036
36.823108288894055
350.86553778278824
341.56001532937853
(0, 1, 2, 4, 6, 8, 5, 0)
341.56001532937853
285.30998564087383
87.94116472882074
352.01156933177674
355.5851803854494
293.04255995502933
103.18921924126201
341.56001532937853
(0, 1, 2, 30, 33, 32, 29, 6, 4, 3, 0)
209.25266950703704
80.13963064879714
67.9068651164943
274.92004687754303
38.534582512735284
45.463327727432706
166.45527800449133
65.39276474508715
36.823108288894055
350.86553778278824
209.25266950703704
(0, 1, 2, 30, 33, 32, 29, 6, 8, 5, 0)
209.25266950703704
80.13963064879714
67.9068651164943
274.92004687754303
38.534582512735284
233.0282837379492
164.9547777098668
355.5851803854494
293.04255995502933
103.18921924126201
209.25266950703704
(0, 1, 2, 30, 33, 32, 46, 43, 40, 38, 29, 6, 4, 3, 0)
209.25266950703704
80.13963064879714
67.9068651164943
157.85036764579326
179.35228006609776
305.8628684459087
50.05212128615715
304.0985891